<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2026%20-%20KD-Trees%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 26: KD-Trees — Indexação Multidimensional e Busca Espacial na Planta de Biodiesel (Grupo 5)

Este caderno implementa uma **KD-Tree ($k$-Dimensional Tree)** do zero em Python e a aplica em dois problemas reais da **Planta Piloto de Produção de Biodiesel em Batelada**:
1. **Monitoramento Multivariável do Reator CSTR `R-200` ($k=4$):** Indexação de estados históricos (`TT-201`, `PT-201`, `LT-201`, `AT-100`) para diagnóstico em tempo real via $k$-NN e auditoria de desvios por *Range Search*.
2. **Aceleração da Primitiva `NEAREST` do `AGV-SAMPLER` ($k=2$):** Benchmark de desempenho da KD-Tree vs. Busca Linear no planejamento de trajetória RRT (Aula 22).

In [ ]:
import math
import time
import heapq
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)
np.random.seed(42)

## 1. Implementação Formal da KD-Tree (Construção Balanceada, $k$-NN e *Range Search*)

In [ ]:
class KDNode:
    def __init__(self, point, label, axis, left=None, right=None):
        self.point = point
        self.label = label
        self.axis = axis
        self.left = left
        self.right = right


class KDTree:
    def __init__(self, points, labels=None):
        self.k = len(points[0]) if points else 0
        if labels is None:
            labels = [None] * len(points)
        paired = list(zip(points, labels))
        self.root = self._build(paired, depth=0)
        self.last_visited_nodes = 0

    def _build(self, paired, depth):
        if not paired:
            return None
        axis = depth % self.k
        paired.sort(key=lambda item: item[0][axis])
        mid = len(paired) // 2
        point, label = paired[mid]
        return KDNode(
            point=point,
            label=label,
            axis=axis,
            left=self._build(paired[:mid], depth + 1),
            right=self._build(paired[mid + 1:], depth + 1),
        )

    @staticmethod
    def _sq_dist(a, b):
        return sum((x - y) ** 2 for x, y in zip(a, b))

    def nearest_neighbor(self, query):
        self.last_visited_nodes = 0
        best = [None, None, float('inf')]  # [point, label, sq_dist]

        def _search(node):
            if node is None:
                return
            self.last_visited_nodes += 1
            d2 = self._sq_dist(query, node.point)
            if d2 < best[2]:
                best[0], best[1], best[2] = node.point, node.label, d2

            axis = node.axis
            diff = query[axis] - node.point[axis]
            near, far = (node.left, node.right) if diff <= 0 else (node.right, node.left)

            _search(near)
            if diff ** 2 < best[2]:
                _search(far)

        _search(self.root)
        return best[0], best[1], math.sqrt(best[2])

    def k_nearest_neighbors(self, query, k_neighbors=5):
        self.last_visited_nodes = 0
        heap = []  # max-heap simulado com distância negativa

        def _search(node):
            if node is None:
                return
            self.last_visited_nodes += 1
            d2 = self._sq_dist(query, node.point)
            if len(heap) < k_neighbors:
                heapq.heappush(heap, (-d2, node.point, node.label))
            elif d2 < -heap[0][0]:
                heapq.heapreplace(heap, (-d2, node.point, node.label))

            axis = node.axis
            diff = query[axis] - node.point[axis]
            near, far = (node.left, node.right) if diff <= 0 else (node.right, node.left)

            _search(near)
            worst_d2 = -heap[0][0] if len(heap) == k_neighbors else float('inf')
            if diff ** 2 < worst_d2:
                _search(far)

        _search(self.root)
        return sorted([(math.sqrt(-neg_d2), pt, lbl) for neg_d2, pt, lbl in heap], key=lambda x: x[0])

    def range_search(self, lower, upper):
        self.last_visited_nodes = 0
        results = []

        def _search(node):
            if node is None:
                return
            self.last_visited_nodes += 1
            if all(lower[i] <= node.point[i] <= upper[i] for i in range(self.k)):
                results.append((node.point, node.label))
            axis = node.axis
            if lower[axis] <= node.point[axis]:
                _search(node.left)
            if upper[axis] >= node.point[axis]:
                _search(node.right)

        _search(self.root)
        return results

## 2. Aplicação 1: Indexação Multivariável do Reator `R-200` ($k = 4$ Dimensões)

Geramos um histórico simulado de $N = 1.200$ amostras de telemetria `[TT-201 (°C), PT-201 (bar), LT-201 (%), AT-100 (ppm)]` distribuídas em 5 regimes operacionais da planta de biodiesel, normalizamos os eixos e validamos a exatidão da KD-Tree contra a busca exaustiva.

In [ ]:
# Geração de histórico SCADA realista (TT-201, PT-201, LT-201, AT-100)
regimes_spec = [
    ('NORMAL_B100',           600, [57.5, 1.05, 65.0,  2.0], [1.2, 0.05, 10.0, 1.0]),
    ('SUB_AQUECIMENTO',       200, [48.0, 1.00, 60.0,  1.5], [2.5, 0.04, 10.0, 0.8]),
    ('PRE_ALERTA_EXOTERMICO', 180, [62.5, 1.75, 65.0,  5.0], [1.0, 0.25,  8.0, 2.0]),
    ('FLASH_BOILING_CRITICO', 100, [67.0, 2.75, 68.0, 12.0], [1.5, 0.25,  7.0, 4.0]),
    ('VAZAMENTO_METANOL',      70, [56.0, 1.05, 55.0, 28.0], [1.5, 0.05, 10.0, 5.0]),
    ('CAVITACAO_AG201',        50, [57.0, 1.02, 11.0,  2.0], [1.5, 0.04,  2.5, 1.0]),
]

raw_points = []
labels = []
for name, count, means, stds in regimes_spec:
    pts = np.random.normal(loc=means, scale=stds, size=(count, 4))
    pts[:, 1] = np.clip(pts[:, 1], 0.8, 4.0)
    pts[:, 2] = np.clip(pts[:, 2], 2.0, 98.0)
    pts[:, 3] = np.clip(pts[:, 3], 0.0, 60.0)
    for row in pts:
        raw_points.append(tuple(round(float(v), 3) for v in row))
        labels.append(name)

arr = np.array(raw_points)
mu = arr.mean(axis=0)
sigma = arr.std(axis=0)
norm_points = [tuple(((row - mu) / sigma).tolist()) for row in arr]

t0 = time.perf_counter()
kd_scada = KDTree(norm_points, labels)
build_ms = (time.perf_counter() - t0) * 1000
print(f'KD-Tree 4D construída com N = {len(norm_points)} amostras em {build_ms:.2f} ms')

# Consulta de telemetria ao vivo: deriva térmica incipiente
query_raw = np.array([63.2, 1.85, 62.0, 4.5])
query_norm = tuple(((query_raw - mu) / sigma).tolist())

pt_nn, lbl_nn, dist_nn = kd_scada.nearest_neighbor(query_norm)
visited_nn = kd_scada.last_visited_nodes

# Validação contra busca exaustiva (Brute-Force)
bf_best = min(
    ((math.sqrt(sum((a - b)**2 for a, b in zip(query_norm, p))), p, l) for p, l in zip(norm_points, labels)),
    key=lambda x: x[0]
)
assert abs(dist_nn - bf_best[0]) < 1e-9, 'Divergência entre KD-Tree e Brute-Force!'

print(f'Telemetria Atual: TT-201={query_raw[0]}°C, PT-201={query_raw[1]} bar, LT-201={query_raw[2]}%, AT-100={query_raw[3]} ppm')
print(f'Vizinho Mais Próximo: Regime={lbl_nn} | Distância Normalizada={dist_nn:.4f}')
print(f'Eficiência de Poda: {visited_nn}/{len(norm_points)} nós visitados ({100*(1 - visited_nn/len(norm_points)):.1f}% podados)')

# Auditoria por Range Search na Zona de Risco Térmico (TT-201 in [61, 68], PT-201 in [1.5, 3.0])
low_raw = np.array([61.0, 1.5, 0.0, 0.0])
high_raw = np.array([68.0, 3.0, 100.0, 60.0])
low_norm = tuple(((low_raw - mu) / sigma).tolist())
high_norm = tuple(((high_raw - mu) / sigma).tolist())

range_hits = kd_scada.range_search(low_norm, high_norm)
print(f'Range Search (Zona de Alerta/Runaway): {len(range_hits)} registros históricos encontrados ({kd_scada.last_visited_nodes} nós visitados).')

## 3. Visualização do Particionamento Espacial 2D (`TT-201` vs. `PT-201`)

In [ ]:
# Subconjunto 2D para visualização nítida dos hiperplanos ortogonais
idx_vis = np.random.choice(len(arr), size=160, replace=False)
pts_2d = [tuple(arr[i, :2].tolist()) for i in idx_vis]
lbl_2d = [labels[i] for i in idx_vis]
kd_2d = KDTree(pts_2d, lbl_2d)

fig, ax = plt.subplots(figsize=(10, 6))
color_map = {
    'NORMAL_B100': '#2ecc71',
    'SUB_AQUECIMENTO': '#3498db',
    'PRE_ALERTA_EXOTERMICO': '#f39c12',
    'FLASH_BOILING_CRITICO': '#e74c3c',
    'VAZAMENTO_METANOL': '#9b59b6',
    'CAVITACAO_AG201': '#1abc9c'
}

def plot_splits(node, xmin, xmax, ymin, ymax, depth=0):
    if node is None or depth > 4:
        return
    x, y = node.point
    if node.axis == 0:
        ax.plot([x, x], [ymin, ymax], color='#94a3b8', linestyle='--', linewidth=max(0.5, 1.8 - 0.3*depth), alpha=0.6)
        plot_splits(node.left, xmin, x, ymin, ymax, depth + 1)
        plot_splits(node.right, x, xmax, ymin, ymax, depth + 1)
    else:
        ax.plot([xmin, xmax], [y, y], color='#64748b', linestyle=':', linewidth=max(0.5, 1.8 - 0.3*depth), alpha=0.6)
        plot_splits(node.left, xmin, xmax, ymin, y, depth + 1)
        plot_splits(node.right, xmin, xmax, y, ymax, depth + 1)

xmin, xmax = 40, 73
ymin, ymax = 0.7, 3.5
plot_splits(kd_2d.root, xmin, xmax, ymin, ymax)

for reg, col in color_map.items():
    sub = [p for p, l in zip(pts_2d, lbl_2d) if l == reg]
    if sub:
        xs, ys = zip(*sub)
        ax.scatter(xs, ys, c=col, label=reg, s=35, edgecolors='white', linewidth=0.5, zorder=3)

# Retângulo de Range Search
rect = patches.Rectangle((61.0, 1.5), 7.0, 1.5, linewidth=2, edgecolor='#dc2626', facecolor='#ef4444', alpha=0.12, label='Range Search (Risco Térmico)')
ax.add_patch(rect)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel('TT-201: Temperatura do Reator R-200 (°C)', fontsize=11)
ax.set_ylabel('PT-201: Pressão do Reator R-200 (bar)', fontsize=11)
ax.set_title('Particionamento KD-Tree (2D) dos Regimes Operacionais do Reator R-200', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=8.5, framealpha=0.95)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Aplicação 2: Aceleração da Primitiva `NEAREST` do `AGV-SAMPLER` (Sinergia com Aula 22 - RRT)

Comparamos o tempo total de execução de $Q = 300$ consultas `NEAREST` em um galpão industrial de $40\text{ m} \times 30\text{ m}$ variando o tamanho $|V|$ da árvore RRT.

In [ ]:
sizes = [100, 500, 1000, 2500, 5000]
t_kd_list, t_bf_list = [], []
n_queries = 300
queries_2d = [(random.uniform(0, 40), random.uniform(0, 30)) for _ in range(n_queries)]

for n_pts in sizes:
    rrt_vertices = [(random.uniform(0, 40), random.uniform(0, 30)) for _ in range(n_pts)]
    kd_agv = KDTree(rrt_vertices)

    t0 = time.perf_counter()
    for q in queries_2d:
        kd_agv.nearest_neighbor(q)
    t_kd_list.append((time.perf_counter() - t0) * 1000)

    t0 = time.perf_counter()
    for q in queries_2d:
        min(rrt_vertices, key=lambda v: (v[0]-q[0])**2 + (v[1]-q[1])**2)
    t_bf_list.append((time.perf_counter() - t0) * 1000)

    print(f'|V| = {n_pts:4d} | KD-Tree: {t_kd_list[-1]:6.2f} ms | Brute-Force: {t_bf_list[-1]:7.2f} ms | Speedup: {t_bf_list[-1]/t_kd_list[-1]:.1f}x')

plt.figure(figsize=(8, 4.5))
plt.plot(sizes, t_bf_list, 'o--', color='#e74c3c', label='Busca Linear O(|V|)')
plt.plot(sizes, t_kd_list, 's-', color='#2563eb', label='KD-Tree 2D O(log |V|)')
plt.xlabel('Número de Vértices na Árvore RRT (|V|)')
plt.ylabel(f'Tempo para {n_queries} consultas NEAREST (ms)')
plt.title('Aceleração do Planejamento RRT do AGV-SAMPLER com KD-Tree')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()